In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import yfinance as yf
import joblib, json, os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import RobustScaler

os.makedirs('model_artifacts', exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Environment ready. Using device: {device}")

✅ Environment ready. Using device: cpu


In [3]:
def get_quant_features(ticker="NVDA", predict_days=5):
    # FIX: Using Ticker().history() avoids the yfinance MultiIndex column bug
    ticker_obj = yf.Ticker(ticker)
    df = ticker_obj.history(start="2015-01-01")
    
    # 1. Standard Returns
    df['Returns'] = df['Close'].pct_change()
    
    # 2. Advanced Technicals
    df['MA20'] = df['Close'].rolling(20).mean()
    df['MA50'] = df['Close'].rolling(50).mean()
    
    # MACD (Moving Average Convergence Divergence)
    ema12 = df['Close'].ewm(span=12, adjust=False).mean()
    ema26 = df['Close'].ewm(span=26, adjust=False).mean()
    df['MACD'] = ema12 - ema26
    
    # Bollinger Bands Width (Measures Volatility squeezed)
    std20 = df['Close'].rolling(20).std()
    df['BB_Width'] = (4 * std20) / df['MA20']
    
    # RSI
    delta = df['Close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
    df['RSI'] = 100 - (100 / (1 + (gain / loss)))
    
    df['Vol_Rel'] = df['Volume'] / df['Volume'].rolling(20).mean()
    
    # 🚨 THE QUANT TARGET: Predicting the N-Day Forward Return (Percentage)
    df['Target_Return'] = (df['Close'].shift(-predict_days) - df['Close']) / df['Close']
    
    return df.dropna()

data = get_quant_features("NVDA", predict_days=5)
print(f"✅ Data processed. Total Rows: {len(data)}")

SEQ_LEN = 30
feature_cols = ['Close', 'Returns', 'MA20', 'MA50', 'MACD', 'BB_Width', 'RSI', 'Vol_Rel']

✅ Data processed. Total Rows: 2795


In [4]:
split_idx = int(len(data) * 0.8)
train_data = data.iloc[:split_idx]
test_data = data.iloc[split_idx:]

# We only need to scale the inputs. 
# The target is a percentage (e.g., -0.02 to 0.05), which is already naturally scaled!
feature_scaler = RobustScaler()
scaled_train_X = feature_scaler.fit_transform(train_data[feature_cols])
scaled_test_X = feature_scaler.transform(test_data[feature_cols])

y_train_raw = train_data['Target_Return'].values
y_test_raw = test_data['Target_Return'].values

In [5]:
def create_sequences(X_data, y_data, seq_len):
    X, y = [], []
    for i in range(len(X_data) - seq_len):
        X.append(X_data[i:(i + seq_len)])
        y.append(y_data[i + seq_len])
    return np.array(X), np.array(y)

X_train_seq, y_train_seq = create_sequences(scaled_train_X, y_train_raw, SEQ_LEN)
X_test_seq, y_test_seq = create_sequences(scaled_test_X, y_test_raw, SEQ_LEN)

X_train_tensor = torch.FloatTensor(X_train_seq).to(device)
y_train_tensor = torch.FloatTensor(y_train_seq).view(-1, 1).to(device)

X_test_tensor = torch.FloatTensor(X_test_seq).to(device)
y_test_tensor = torch.FloatTensor(y_test_seq).view(-1, 1).to(device)

print(f"✅ Train Input Shape: {X_train_tensor.shape}")

✅ Train Input Shape: torch.Size([2206, 30, 8])


In [6]:
class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super(Attention, self).__init__()
        self.attention = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, x):
        # x shape: (batch_size, seq_len, hidden_dim)
        scores = self.attention(x).squeeze(-1) # (batch, seq_len)
        weights = torch.softmax(scores, dim=-1).unsqueeze(-1) # (batch, seq_len, 1)
        context = torch.sum(x * weights, dim=1) # (batch, hidden_dim)
        return context

class QuantSageModel(nn.Module):
    def __init__(self, n_features):
        super(QuantSageModel, self).__init__()
        
        # 1. Feature Extraction
        self.cnn = nn.Conv1d(in_channels=n_features, out_channels=64, kernel_size=3, padding=1)
        
        # 2. Bidirectional Temporal Processing (Reads past to present AND present to past)
        self.bilstm = nn.LSTM(input_size=64, hidden_size=64, batch_first=True, bidirectional=True)
        
        # 3. Attention (Focuses on the most critical days in the sequence)
        # Hidden dim is 128 because BiLSTM concatenates forward (64) and backward (64) states
        self.attention = Attention(128) 
        
        self.dropout = nn.Dropout(0.4)
        self.fc1 = nn.Linear(128, 32)
        self.fc2 = nn.Linear(32, 1) 
        self.relu = nn.ReLU()

    def forward(self, x):
        x = x.transpose(1, 2)
        x = self.relu(self.cnn(x))
        x = x.transpose(1, 2)
        
        lstm_out, _ = self.bilstm(x)
        
        # Apply attention to LSTM outputs
        attn_out = self.attention(lstm_out)
        
        x = self.dropout(attn_out)
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        
        # Outputs predicted percentage return
        return self.fc2(x)

model = QuantSageModel(len(feature_cols)).to(device)
print(model)

QuantSageModel(
  (cnn): Conv1d(8, 64, kernel_size=(3,), stride=(1,), padding=(1,))
  (bilstm): LSTM(64, 64, batch_first=True, bidirectional=True)
  (attention): Attention(
    (attention): Linear(in_features=128, out_features=1, bias=False)
  )
  (dropout): Dropout(p=0.4, inplace=False)
  (fc1): Linear(in_features=128, out_features=32, bias=True)
  (fc2): Linear(in_features=32, out_features=1, bias=True)
  (relu): ReLU()
)


In [7]:
# HuberLoss acts like MSE for small errors, but MAE for large errors (outliers like market crashes)
criterion = nn.HuberLoss(delta=1.0) 
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4) # AdamW is better for regularization

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [8]:
epochs = 40
best_val_loss = float('inf')
patience = 7
patience_counter = 0

print("🚀 Training Institutional Quant Model...")
for epoch in range(epochs):
    model.train()
    train_loss = 0.0
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    
    avg_train_loss = train_loss / len(train_loader)
    
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            val_loss += loss.item()
            
    avg_val_loss = val_loss / len(test_loader)
    
    if (epoch+1) % 5 == 0 or epoch == 0:
        print(f"Epoch [{epoch+1}/{epochs}] | Train Loss: {avg_train_loss:.5f} | Val Loss: {avg_val_loss:.5f}")
        
    # Early Stopping Logic
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), 'model_artifacts/best_quant_model.pth')
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"🛑 Early stopping triggered at epoch {epoch+1}. Restoring best model.")
            break

print(f"✅ Training Complete. Best Validation Loss: {best_val_loss:.5f}")

🚀 Training Institutional Quant Model...
Epoch [1/40] | Train Loss: 0.00234 | Val Loss: 0.00242
Epoch [5/40] | Train Loss: 0.00218 | Val Loss: 0.00222
Epoch [10/40] | Train Loss: 0.00217 | Val Loss: 0.00211
🛑 Early stopping triggered at epoch 10. Restoring best model.
✅ Training Complete. Best Validation Loss: 0.00203


In [10]:
joblib.dump(feature_scaler, 'model_artifacts/feature_scaler.pkl')

metadata = {
    "feature_cols": feature_cols,
    "seq_len": SEQ_LEN,
    "predict_days": 5, 
    "framework": "pytorch",
    "type": "return_regression" # Predicting percentages now
}

with open('model_artifacts/metadata.json', 'w') as f:
    json.dump(metadata, f)

print("✅ Quant Model, Scaler, and Metadata successfully saved!")

✅ Quant Model, Scaler, and Metadata successfully saved!
